In [15]:
# =========================================================
# CELL 1 — Imports + Reproducibility
# =========================================================

import os
import re
import math
import ast
import json
import pickle
import random
from collections import defaultdict

import numpy as np
import pandas as pd

import torch
import torch.nn as nn
import torch.nn.functional as F
from tqdm.auto import tqdm

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)

torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False


In [16]:
# =========================================================
# CELL 1.5 — Device
# =========================================================

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", device)


Device: cuda


In [17]:
# =========================================================
# CELL 2 — Load CSVs + Embeddings
# =========================================================

MAIN_PATH   = r"/teamspace/studios/this_studio/datasets/main_mixed_df.csv"
DIALOG_PATH = r"/teamspace/studios/this_studio/datasets/dialogue.csv"
DOCS_PATH   = r"/teamspace/studios/this_studio/datasets/documents.csv"
RESP_PATH   = r"/teamspace/studios/this_studio/datasets/response.csv"
SUMM_PATH   = r"/teamspace/studios/this_studio/datasets/summaries.csv"

EMB_DIALOG_PATH = r"/teamspace/studios/this_studio/embeddings/dialog_embeddings.pkl"
EMB_DOCS_PATH   = r"/teamspace/studios/this_studio/embeddings/doc_embeddings.pkl"
EMB_RESP_PATH   = r"/teamspace/studios/this_studio/embeddings/response_embeddings.pkl"
EMB_SUMM_PATH   = r"/teamspace/studios/this_studio/embeddings/summ_embeddings.pkl"

main_df  = pd.read_csv(MAIN_PATH)
dialog_df= pd.read_csv(DIALOG_PATH)
docs_df  = pd.read_csv(DOCS_PATH)
resp_df  = pd.read_csv(RESP_PATH)
summ_df  = pd.read_csv(SUMM_PATH)

def load_embeddings(file_path):
    with open(file_path, "rb") as f:
        return pickle.load(f)

emb_dialog = load_embeddings(EMB_DIALOG_PATH)
emb_docs   = load_embeddings(EMB_DOCS_PATH)
emb_resp   = load_embeddings(EMB_RESP_PATH)
emb_summ   = load_embeddings(EMB_SUMM_PATH)

print("Loaded main_df columns:", list(main_df.columns))
print("Rows:", len(main_df))
print("Sample row:\n", main_df.iloc[0])


Loaded main_df columns: ['Unnamed: 0', 'userid', 'target_user_id', 'task', 'text_nodes', 'interactions', 'query_text_id', 'target_text_id', 'split']
Rows: 113274
Sample row:
 Unnamed: 0                                                        0
userid                                                      U203312
target_user_id                                              U203312
task                                           next_item_prediction
text_nodes        ['d-100554', 'd-114190', 'd-58081', 'd-40294',...
interactions      ['pos', 'pos', 'pos', 'pos', 'neg', 'pos', 'po...
query_text_id                                               d-57729
target_text_id                                              d-57729
split                                                         train
Name: 0, dtype: object


In [18]:
# =========================================================
# CELL 3 — Core Helpers (sequence parse, canonical ids, null checks)
# =========================================================

CANON_RE = re.compile(r"^(dl|d|r|s)-\d+$", re.IGNORECASE)

def is_nullish(x):
    if x is None:
        return True
    if isinstance(x, float) and np.isnan(x):
        return True
    s = str(x).strip().lower()
    return s in {"", "null", "none", "nan"}

def normalize_id(x):
    return str(x).strip()

def to_canonical(prefix: str, raw_id):
    s = str(raw_id).strip()
    if CANON_RE.match(s):
        return s
    return f"{prefix}-{s}"

def parse_serialized_sequence(x):
    """
    Accepts:
      - Python list
      - '["a","b"]'
      - '[a,b]'
    Returns: list[str]
    """
    if x is None or (isinstance(x, float) and np.isnan(x)):
        return []
    if isinstance(x, list):
        return [str(t).strip() for t in x if str(t).strip() != ""]
    s = str(x).strip()
    if not (s.startswith("[") and s.endswith("]")):
        return []
    try:
        v = ast.literal_eval(s)
        if isinstance(v, list):
            return [str(t).strip() for t in v if str(t).strip() != ""]
    except Exception:
        pass
    items = [t.strip().strip('"').strip("'") for t in s[1:-1].split(",")]
    return [t for t in items if t]

def fix_and_align_sequences(nodes, actions, max_len=100):
    nodes = list(nodes)
    actions = list(actions)
    L = min(len(nodes), len(actions), max_len)
    return nodes[:L], actions[:L]


In [19]:
# =========================================================
# CELL 4 — Build id2text (robust fallback)
# =========================================================

def infer_id_text_map(df: pd.DataFrame):
    """
    Attempts to infer (id_column, text_column) from a dataframe.
    Returns: dict[str -> str]
    If cannot infer, returns empty dict.
    """
    cols = [c.lower() for c in df.columns]
    df_cols = list(df.columns)

    # Candidate id columns (ordered)
    id_candidates = []
    for c in df_cols:
        cl = c.lower()
        if "text_id" in cl or cl.endswith("_id") or cl == "id":
            id_candidates.append(c)

    # Candidate text columns (ordered)
    text_candidates = []
    for c in df_cols:
        cl = c.lower()
        if cl in {"text", "content", "body", "summary", "response", "utterance"}:
            text_candidates.append(c)
        elif "text" in cl or "content" in cl or "body" in cl or "summary" in cl:
            text_candidates.append(c)

    # Try pairs
    for idc in id_candidates:
        for tc in text_candidates:
            try:
                sub = df[[idc, tc]].dropna()
                if len(sub) == 0:
                    continue
                m = {}
                for rid, txt in zip(sub[idc].astype(str), sub[tc].astype(str)):
                    rid = rid.strip()
                    if rid == "":
                        continue
                    if rid not in m:
                        m[rid] = txt
                if len(m) > 0:
                    return m
            except Exception:
                continue

    return {}

id2text = defaultdict(str)

for _df in [dialog_df, docs_df, resp_df, summ_df]:
    m = infer_id_text_map(_df)
    for k, v in m.items():
        if k not in id2text:
            id2text[k] = v

print("id2text size:", len(id2text))


id2text size: 13966


In [20]:
# =========================================================
# CELL 5 — Embedding Resolver for canonical ids
# =========================================================

class EmbLookup:
    @staticmethod
    def lookup(emb_dict, key):
        if key in emb_dict:
            return emb_dict[key]
        s = str(key)
        if s in emb_dict:
            return emb_dict[s]
        nums = re.findall(r"\d+", s)
        if nums:
            n = nums[-1]
            if n in emb_dict:
                return emb_dict[n]
            try:
                ni = int(n)
                if ni in emb_dict:
                    return emb_dict[ni]
            except Exception:
                pass
        return None

class CanonicalEmbeddingResolver:
    def __init__(self, emb_dialog, emb_docs, emb_resp, emb_summ, d=768):
        self.emb_dialog = emb_dialog
        self.emb_docs = emb_docs
        self.emb_resp = emb_resp
        self.emb_summ = emb_summ
        self.d = d

    def get(self, cid: str) -> np.ndarray:
        cid = normalize_id(cid)
        prefix = cid.split("-")[0].lower()

        if prefix == "dl":
            v = EmbLookup.lookup(self.emb_dialog, cid)
        elif prefix == "d":
            v = EmbLookup.lookup(self.emb_docs, cid)
        elif prefix == "r":
            v = EmbLookup.lookup(self.emb_resp, cid)
        elif prefix == "s":
            v = EmbLookup.lookup(self.emb_summ, cid)
        else:
            v = None

        if v is None:
            return np.zeros((self.d,), dtype=np.float32)

        v = np.asarray(v, dtype=np.float32)
        if v.shape[-1] != self.d:
            raise ValueError(f"Embedding dim mismatch for {cid}: got {v.shape}, expected ({self.d},)")
        return v

emb_resolver = CanonicalEmbeddingResolver(emb_dialog, emb_docs, emb_resp, emb_summ, d=768)
print("Embedding resolver ready.")


Embedding resolver ready.


In [21]:
# =========================================================
# CELL 6 — Task Names + Task IDs
# =========================================================

assert "task" in main_df.columns, "main_df must contain 'task'"

task_names = sorted(main_df["task"].dropna().astype(str).unique())
task2idx = {t: i for i, t in enumerate(task_names)}
idx2task = {i: t for t, i in task2idx.items()}

main_df["task_id"] = main_df["task"].astype(str).map(task2idx)
assert main_df["task_id"].isna().sum() == 0, "Unmapped task found"

n_tasks = len(task2idx)
print("Tasks:", task2idx)


Tasks: {'dialogue_generation': 0, 'next_item_prediction': 1, 'next_user_prediction': 2, 'personalized_summarization': 3}


In [22]:
# =========================================================
# CELL 7 — Build ACTION vocab (all rows) and NODE vocab (all rows)
# =========================================================

all_actions = set()
for seq in main_df["interactions"].dropna():
    all_actions.update(parse_serialized_sequence(seq))

actions2idx = {a: i for i, a in enumerate(sorted(all_actions))}
idx2action  = {i: a for a, i in actions2idx.items()}

node_ids = set()
for seq in main_df["text_nodes"].dropna():
    for nid in parse_serialized_sequence(seq):
        if not is_nullish(nid):
            node_ids.add(normalize_id(nid))

for col in ["query_text_id", "target_text_id"]:
    if col in main_df.columns:
        for x in main_df[col].dropna():
            if not is_nullish(x):
                node_ids.add(normalize_id(x))

node2idx = {nid: i for i, nid in enumerate(sorted(node_ids))}
idx2node = {i: nid for nid, i in node2idx.items()}

print("VOCAB SIZES:")
print("  actions:", len(actions2idx))
print("  nodes  :", len(node2idx))


VOCAB SIZES:
  actions: 4
  nodes  : 90496


In [23]:
# =========================================================
# CELL 8 — Build USER vocab ONLY from next_user_prediction rows
# =========================================================

USER_TASK = "next_user_prediction"

user_rows = main_df[main_df["task"].astype(str).str.strip() == USER_TASK]
unique_users_small = sorted(user_rows["userid"].dropna().astype(str).unique())
unique_users_small = [""] + [u for u in unique_users_small if u != ""]

user2idx_small = {u: i for i, u in enumerate(unique_users_small)}
idx2user_small = {i: u for u, i in user2idx_small.items()}

print("User vocab size (next_user_prediction only):", len(user2idx_small))


User vocab size (next_user_prediction only): 4527


In [24]:
# =========================================================
# CELL 9 — Minimal safe definitions for item/generation symbols
# (No assumptions: derived from available task_names; otherwise safe fallbacks)
# =========================================================

# ITEM_TASK: if a next-item task exists in your task list, use that; else use a sentinel not present
ITEM_TASK = "next_item_prediction" if "next_item_prediction" in task2idx else "__no_item_task__"

# GEN_TASKS_WORD: include tasks that look like generation tasks if they exist; else empty
GEN_TASKS_WORD = set()
for t in task_names:
    tl = t.lower()
    if ("generation" in tl) or ("summar" in tl) or ("dialog" in tl and "generation" in tl):
        GEN_TASKS_WORD.add(t)

# word2idx / id2words are required by your builder, even if unused
word2idx = {"<pad>": 0}
id2words = defaultdict(list)

# item2idx is required by your builder; keep minimal safe
item2idx = {"": 0}

print("ITEM_TASK:", ITEM_TASK)
print("GEN_TASKS_WORD:", GEN_TASKS_WORD)
print("word2idx size:", len(word2idx))
print("item2idx size:", len(item2idx))


ITEM_TASK: next_item_prediction
GEN_TASKS_WORD: {'dialogue_generation', 'personalized_summarization'}
word2idx size: 1
item2idx size: 1


In [25]:
# =========================================================
# CELL 10 — Split train/test
# =========================================================

assert "split" in main_df.columns, "main_df must contain 'split' with train/test."

main_df["split"] = main_df["split"].astype(str).str.strip().str.lower()
train_df = main_df[main_df["split"] == "train"].reset_index(drop=True)
test_df  = main_df[main_df["split"] == "test"].reset_index(drop=True)

print("Train size:", len(train_df), "| Test size:", len(test_df))
print("Task counts (train):\n", train_df["task"].value_counts())
print("Task counts (test):\n", test_df["task"].value_counts())


Train size: 78274 | Test size: 35000
Task counts (train):
 task
next_item_prediction          34630
personalized_summarization    34630
next_user_prediction           4507
dialogue_generation            4507
Name: count, dtype: int64
Task counts (test):
 task
next_item_prediction          15000
personalized_summarization    15000
next_user_prediction           2500
dialogue_generation            2500
Name: count, dtype: int64


In [26]:
# =========================================================
# CELL 11 — Batch Builder (row -> tensors) with TASK-WISE masks
# =========================================================

MAX_T = 60
D_MODEL = 768
IGNORE_INDEX = -100

def build_multi_hot_words(text_id: str, device):
    y = torch.zeros(len(word2idx), device=device)
    words = id2words.get(text_id, [])
    for w in words:
        j = word2idx[w] if w in word2idx else 0
        y[j] = 1.0
    return y, words

def safe_user_label(userid: str):
    userid = str(userid).strip()
    if userid in user2idx_small:
        return user2idx_small[userid]
    return 0

def safe_item_label(itemid: str):
    itemid = normalize_id(itemid)
    if itemid in item2idx:
        return item2idx[itemid]
    return 0

def build_row_tensors(row, device):
    task = str(row["task"]).strip()
    task_id = int(row["task_id"])

    actions = parse_serialized_sequence(row["interactions"])
    nodes   = parse_serialized_sequence(row["text_nodes"])

    if len(actions) == 0 or len(nodes) == 0:
        return {"ok": False, "reason": "empty_sequence", "task": task}

    nodes, actions = fix_and_align_sequences(nodes, actions, max_len=MAX_T)
    if len(nodes) < 3:
        return {"ok": False, "reason": "too_short_sequence", "task": task}

    T = len(nodes)

    try:
        action_ids = [actions2idx[a] for a in actions]
    except KeyError:
        return {"ok": False, "reason": "unknown_action_token", "task": task}

    action_labels_full = torch.tensor(action_ids, device=device).unsqueeze(0)  # (1,T)
    action_onehot = F.one_hot(action_labels_full, num_classes=len(actions2idx)).float()  # (1,T,A)

    nodes_norm = [normalize_id(n) for n in nodes]
    try:
        node_ids = [node2idx[n] for n in nodes_norm]
    except KeyError:
        return {"ok": False, "reason": "unknown_node_token", "task": task}

    node_labels_full = torch.tensor(node_ids, device=device).unsqueeze(0)  # (1,T)

    e_seq = torch.tensor(
        np.stack([emb_resolver.get(n) for n in nodes_norm], axis=0),
        device=device
    ).unsqueeze(0)  # (1,T,768)

    qid = normalize_id(row["query_text_id"]) if "query_text_id" in row else nodes_norm[-2]
    tid = normalize_id(row["target_text_id"]) if "target_text_id" in row else nodes_norm[-1]

    q_text_emb = torch.tensor(emb_resolver.get(qid), device=device).unsqueeze(0)  # (1,768)

    q_text = str(id2text.get(qid)).strip()
    t_text = str(id2text.get(tid)).strip()

    task_labels = torch.tensor([task_id], device=device)

    if task == ITEM_TASK:
        keep_len = max(1, T - 2)
    else:
        keep_len = T

    action_labels = action_labels_full.clone()
    node_labels   = node_labels_full.clone()

    if keep_len < T:
        action_labels[0, keep_len:] = IGNORE_INDEX
        node_labels[0, keep_len:]   = IGNORE_INDEX

    user_mask = torch.tensor([1.0 if task == USER_TASK else 0.0], device=device)
    user_labels = torch.tensor([safe_user_label(row["userid"])], device=device)

    item_mask = torch.tensor([1.0 if task == ITEM_TASK else 0.0], device=device)
    item_target_id = safe_item_label(tid)
    item_labels = torch.tensor([item_target_id], device=device)

    word_mask = torch.tensor([1.0 if task in GEN_TASKS_WORD else 0.0], device=device)

    word_targets = torch.zeros(1, len(word2idx), device=device)
    gold_words = []
    if task in GEN_TASKS_WORD:
        y, gold_words = build_multi_hot_words(tid, device=device)
        word_targets[0] = y

    return {
        "ok": True,
        "task": task,
        "T": T,

        "e_seq": e_seq,
        "action_onehot": action_onehot,
        "action_labels": action_labels,
        "node_labels": node_labels,

        "task_labels": task_labels,
        "q_text_emb": q_text_emb,

        "user_labels": user_labels,
        "user_mask": user_mask,

        "item_labels": item_labels,
        "item_mask": item_mask,

        "word_targets": word_targets,
        "word_mask": word_mask,
        "gold_words": gold_words,

        "qid": qid,
        "tid": tid,
        "q_text": q_text,
        "t_text": t_text,

        "userid_raw": str(row["userid"]).strip(),
    }


In [27]:
# =========================================================
# CELL 12 — FocalCELoss (required by encoder)
# =========================================================

class FocalCELoss(nn.Module):
    def __init__(self, gamma=2.0, reduction="mean"):
        super().__init__()
        self.gamma = gamma
        self.reduction = reduction

    def forward(self, logits, targets):
        # logits: (B,C), targets: (B,)
        ce = F.cross_entropy(logits, targets, reduction="none")
        pt = torch.exp(-ce)
        loss = ((1 - pt) ** self.gamma) * ce

        if self.reduction == "mean":
            return loss.mean()
        if self.reduction == "sum":
            return loss.sum()
        return loss


In [28]:
# =========================================================
# CELL 13 — UniHis SSM Encoder (DROP-IN replacement)
# MODIFICATION SPEC (as you requested):
#   - SSM processes only up to length-2 (i.e., indices 0..T-3)
#   - Then tri-SSM mixing + fusion + task gating -> h_final_minus2
#   - Then h_final_minus1 = W_step(h_final_minus2)
#   - Then h_final = MLP_step(h_final_minus1)
#   - Final-step losses computed using h_final to predict:
#       * action at last index (T-1)
#       * node at last index (T-1)
#   - Sequence losses (action/node) remain for processed steps 0..T-3 only
# =========================================================

class UniHisSSMEncoder(nn.Module):
    def __init__(
        self,
        vocab_sizes,
        d_model=768,
        d_state=64,
        n_tasks=8,
        d_qtext=768,
        known_task=False
    ):
        super().__init__()
        self.d_state = d_state
        self.known_task = known_task

        # Sequence CE for processed part
        self.criterion = nn.CrossEntropyLoss(ignore_index=IGNORE_INDEX)

        # Task CE (focal)
        self.task_criterion = FocalCELoss(gamma=2.0, reduction="mean")

        # ========= SSM EIGEN PARAMS =========
        def init_eig():
            Q = nn.Parameter(torch.randn(d_state, d_state) * 0.02)
            Lambda = nn.Parameter(torch.empty(d_state).uniform_(-1.0, -0.1))
            return Q, Lambda

        self.Q_L, self.Lambda_L = init_eig()
        self.Q_S, self.Lambda_S = init_eig()
        self.Q_E, self.Lambda_E = init_eig()

        self.B = nn.Parameter(torch.randn(d_state, d_model))

        self.tau_S = nn.Parameter(torch.tensor(1.0))
        self.tau_E = nn.Parameter(torch.tensor(1.0))

        # ========= ACTION → MOOD =========
        self.action_proj = nn.Linear(vocab_sizes["actions"], d_model)

        self.MLP_mood = nn.Sequential(
            nn.Linear(d_model, d_model // 2),
            nn.ReLU(),
            nn.Linear(d_model // 2, d_model)
        )

        self.W_a = nn.Linear(d_model, d_state)

        # ========= MIXERS =========
        self.Mixer_L = nn.Linear(d_state, d_state)
        self.Mixer_S = nn.Linear(d_state, d_state)
        self.Mixer_E = nn.Linear(d_state, d_state)

        self.W_fuse = nn.Linear(3 * d_state, 3)
        self.Mixer_Fuse = nn.Linear(d_state, d_state)

        # ========= HEADS for processed steps =========
        self.action_head = nn.Linear(3 * d_state, vocab_sizes["actions"])
        self.node_head   = nn.Linear(3 * d_state, vocab_sizes["nodes"])

        # ========= TASK =========
        self.Embed_Q = nn.Linear(d_qtext, d_state)
        self.W_task  = nn.Linear(2 * d_state, n_tasks)

        self.W_hyp = nn.Linear(2 * d_state, d_state)
        self.W_theta = nn.ModuleList([
            nn.Linear(d_state, 2 * d_state) for _ in range(n_tasks)
        ])

        # ========= NEW: step prediction heads =========
        self.W_step = nn.Linear(d_state, d_state)  # h_{T-1} = W_step(h_{T-2})
        self.MLP_step = nn.Sequential(
            nn.Linear(d_state, d_state),
            nn.ReLU(),
            nn.Linear(d_state, d_state)
        )  # h_T = MLP_step(h_{T-1})

        # ========= NEW: last-step prediction heads =========
        self.action_last_head = nn.Linear(d_state, vocab_sizes["actions"])
        self.node_last_head   = nn.Linear(d_state, vocab_sizes["nodes"])

    def make_A(self, Q, Lambda_diag):
        Q_inv = torch.linalg.inv(Q)
        return Q @ torch.diag(Lambda_diag) @ Q_inv

    def forward(
        self,
        e_seq,             # (B, T, d_model)
        action_onehot,     # (B, T, |A|)
        action_labels,     # (B, T) (may contain IGNORE_INDEX)
        node_labels,       # (B, T) (may contain IGNORE_INDEX)
        task_labels,       # (B,)
        q_text_emb         # (B, d_qtext)
    ):
        B, T, _ = e_seq.shape
        device = e_seq.device

        # We require at least 3 to have a (T-2) stage and last-step supervision
        # build_row_tensors already enforces len(nodes) >= 3
        # processed length = T-2 means last processed index = T-3
        T_proc = max(0, T - 2)

        # ========= INITIAL STATES =========
        xL = torch.zeros(B, self.d_state, device=device)
        xS = torch.zeros(B, self.d_state, device=device)
        xE = torch.zeros(B, self.d_state, device=device)

        act_logits_proc = []
        node_logits_proc = []

        # ========= SSM MATRICES =========
        A_L = self.make_A(self.Q_L, torch.exp(self.Lambda_L))

        eps_S = torch.sigmoid(self.tau_S)
        k = torch.log((1 - eps_S) / eps_S) / torch.abs(self.Lambda_S)
        A_S = self.make_A(self.Q_S, torch.sigmoid(-k * self.Lambda_S))

        eta = (torch.exp(-1.0 / self.tau_E) - 1.0) / self.Lambda_E
        A_E = self.make_A(self.Q_E, 1.0 + eta * self.Lambda_E)

        # ========= SEQUENCE LOOP (ONLY up to T-2) =========
        # This iterates t = 0..T-3 (length = T-2)
        for t in range(T_proc):
            e_t = e_seq[:, t]                      # (B, d_model)
            a_t = action_onehot[:, t, :]           # (B, |A|)

            a_emb = self.action_proj(a_t)
            mood_vec = self.MLP_mood(a_emb)
            mood = torch.sigmoid(self.W_a(mood_vec))  # (B,d_state)

            u_t = torch.bmm(
                (self.B.unsqueeze(0) * mood.unsqueeze(2)),
                e_t.unsqueeze(2)
            ).squeeze(2)  # (B,d_state)

            xL = torch.matmul(xL, A_L.T) + u_t
            xS = torch.matmul(xS, A_S.T) + u_t
            xE = torch.matmul(xE, A_E.T) + u_t

            x_cat = torch.cat([xL, xS, xE], dim=-1)  # (B,3d_state)
            act_logits_proc.append(self.action_head(x_cat))
            node_logits_proc.append(self.node_head(x_cat))

        # Stack processed logits (may be empty if T_proc==0, but T>=3 => T_proc>=1)
        act_logits_proc  = torch.stack(act_logits_proc, dim=1)     # (B, T-2, |A|)
        node_logits_proc = torch.stack(node_logits_proc, dim=1)    # (B, T-2, |V|)

        # ========= HISTORY FUSION (on xL,xS,xE at time T-2) =========
        yL = self.Mixer_L(xL)
        yS = self.Mixer_S(xS)
        yE = self.Mixer_E(xE)

        w = F.softmax(self.W_fuse(torch.cat([xL, xS, xE], dim=-1)), dim=-1)
        h_hist = self.Mixer_Fuse(
            w[:, 0:1] * yL +
            w[:, 1:2] * yS +
            w[:, 2:3] * yE
        )

        # ========= TASK CONDITIONING (produces h_final_minus2) =========
        q_emb = self.Embed_Q(q_text_emb)
        z_switch = torch.cat([h_hist, q_emb], dim=-1)

        task_logits = self.W_task(z_switch)
        P_task = F.softmax(task_logits, dim=-1)
        loss_task = self.task_criterion(task_logits, task_labels)

        h_hyp = F.relu(self.W_hyp(z_switch))

        if self.known_task:
            theta = self.W_theta[task_labels[0]](h_hyp)
            g, b = theta.chunk(2, dim=-1)
            h_final_minus2 = torch.sigmoid(g) * h_hist + b
        else:
            h_final_minus2 = 0.0
            for i in range(len(self.W_theta)):
                theta_i = self.W_theta[i](h_hyp)
                g_i, b_i = theta_i.chunk(2, dim=-1)
                h_final_minus2 = h_final_minus2 + P_task[:, i:i+1] * (
                    torch.sigmoid(g_i) * h_hist + b_i
                )

        # ========= NEW: TWO-STEP FORWARD PREDICTIONS =========
        h_final_minus1 = self.W_step(h_final_minus2)     # learned linear
        h_final = self.MLP_step(h_final_minus1)          # learned MLP

        # ========= LOSSES =========
        # (A) processed sequence losses: compare logits_proc against labels for indices 0..T-3
        #     labels slice length = T-2
        action_labels_proc = action_labels[:, :T_proc]
        node_labels_proc   = node_labels[:, :T_proc]

        loss_action_proc = self.criterion(
            act_logits_proc.reshape(-1, act_logits_proc.size(-1)),
            action_labels_proc.reshape(-1)
        )

        loss_node_proc = self.criterion(
            node_logits_proc.reshape(-1, node_logits_proc.size(-1)),
            node_labels_proc.reshape(-1)
        )

        # (B) final-step loss: predict last token (index T-1) using h_final
        #     If last label is IGNORE_INDEX, skip by returning 0 for that component.
        last_action_label = action_labels[:, -1]  # (B,)
        last_node_label   = node_labels[:, -1]    # (B,)

        act_last_logits = self.action_last_head(h_final)  # (B, |A|)
        node_last_logits= self.node_last_head(h_final)    # (B, |V|)

        # Compute last losses with masking (IGNORE_INDEX handling)
        loss_action_last = torch.tensor(0.0, device=device)
        loss_node_last   = torch.tensor(0.0, device=device)

        valid_act = (last_action_label != IGNORE_INDEX)
        valid_node = (last_node_label != IGNORE_INDEX)

        if valid_act.any():
            loss_action_last = F.cross_entropy(act_last_logits[valid_act], last_action_label[valid_act])
        if valid_node.any():
            loss_node_last = F.cross_entropy(node_last_logits[valid_node], last_node_label[valid_node])

        # Total action/node losses = processed + last-step
        loss_action = loss_action_proc + loss_action_last
        loss_node   = loss_node_proc + loss_node_last

        return {
            "loss_action": loss_action,
            "loss_node": loss_node,
            "loss_task": loss_task,

            "act_logits_proc": act_logits_proc,
            "node_logits_proc": node_logits_proc,
            "act_last_logits": act_last_logits,
            "node_last_logits": node_last_logits,

            "task_logits": task_logits,
            "P_task": P_task,

            "h_final_minus2": h_final_minus2,
            "h_final_minus1": h_final_minus1,
            "h_final": h_final
        }


In [29]:
# =========================================================
# CELL 14 — User Bilinear Ranker
# =========================================================

class UserBilinearRanker(nn.Module):
    def __init__(self, d_state, n_users):
        super().__init__()
        self.W = nn.Parameter(torch.randn(d_state, d_state) * 0.02)
        self.U = nn.Embedding(n_users, d_state)
        nn.init.normal_(self.U.weight, std=0.02)

    def scores_all(self, h):
        return (h @ self.W) @ self.U.weight.T   # (B, n_users)

    def scores_subset(self, h, user_ids):
        hw = h @ self.W
        Usub = self.U(user_ids)   # (B,K,d)
        return torch.einsum("bd,bkd->bk", hw, Usub)


In [30]:
# =========================================================
# CELL 15 — Sampled InfoNCE User Loss
# =========================================================

def sampled_user_loss(ranker, h, gold_users, n_neg=64):
    B = h.size(0)
    device = h.device

    neg = torch.randint(0, ranker.U.num_embeddings, (B, n_neg), device=device)
    gold = gold_users.unsqueeze(1)
    neg = torch.where(neg == gold, (neg + 1) % ranker.U.num_embeddings, neg)

    cand = torch.cat([gold, neg], dim=1)      # (B, 1+n_neg)
    logits = ranker.scores_subset(h, cand)    # (B, 1+n_neg)
    y = torch.zeros(B, dtype=torch.long, device=device)
    return F.cross_entropy(logits, y)


In [31]:
# =========================================================
# CELL 16 — Full Model Wrapper
# =========================================================

class UniHisUserModel(nn.Module):
    def __init__(self, encoder, n_users):
        super().__init__()
        self.encoder = encoder
        self.user_ranker = UserBilinearRanker(encoder.d_state, n_users)

    def forward(self, batch):
        enc = self.encoder(
            e_seq=batch["e_seq"],
            action_onehot=batch["action_onehot"],
            action_labels=batch["action_labels"],
            node_labels=batch["node_labels"],
            task_labels=batch["task_labels"],
            q_text_emb=batch["q_text_emb"]
        )

        h = enc["h_final"]  # use your new h_final (after W + MLP)

        loss_user = torch.tensor(0.0, device=h.device)
        if float(batch["user_mask"].item()) > 0:
            loss_user = sampled_user_loss(self.user_ranker, h, batch["user_labels"])

        loss = enc["loss_action"] + enc["loss_node"] + enc["loss_task"] + loss_user

        return {
            "loss": loss,
            "loss_user": loss_user,
            "enc": enc
        }


In [32]:
# =========================================================
# CELL 17 — Instantiate Model + Optimizer
# =========================================================

encoder = UniHisSSMEncoder(
    vocab_sizes={"actions": len(actions2idx), "nodes": len(node2idx)},
    d_model=768,
    d_state=64,
    n_tasks=n_tasks,
    d_qtext=768,
    known_task=False
).to(device)

model = UniHisUserModel(encoder, n_users=len(user2idx_small)).to(device)

optimizer = torch.optim.AdamW(model.parameters(), lr=3e-4)


In [80]:
# =========================================================
# CELL 18 — Force-Resume Training (Ignore RNG, with Validation)
# =========================================================

EPOCHS = 125
NEG_K = 99

CKPT_DIR = "checkpoints_user_only"
os.makedirs(CKPT_DIR, exist_ok=True)

def ckpt_path(epoch, row):
    return os.path.join(CKPT_DIR, f"epoch{epoch:03d}_row{row:06d}.pt")

def find_latest_ckpt():
    files = [f for f in os.listdir(CKPT_DIR) if f.endswith(".pt")]
    if not files:
        return None
    def key(f):
        m = re.match(r"epoch(\d+)_row(\d+)\.pt", f)
        return (int(m.group(1)), int(m.group(2)))
    files.sort(key=key)
    return files[-1]

# -------- Resume model + optimizer only --------
start_epoch = 15   # default hard-start
start_row = 0

latest = find_latest_ckpt()
if latest is not None:
    ck = torch.load(os.path.join(CKPT_DIR, latest), map_location=device, weights_only=False)
    model.load_state_dict(ck["model"])
    optimizer.load_state_dict(ck["optim"])
    start_epoch = max(start_epoch, ck["epoch"])
    start_row = ck["row"] + 1 if ck["epoch"] >= 15 else 0
    print(f"FORCE resume from {latest} (RNG ignored)")
else:
    print("Starting fresh from epoch 15")

# -------- Build training set --------
user_df = train_df[train_df["task"].astype(str).str.strip() == USER_TASK]
item_df = train_df[train_df["task"].astype(str).str.strip() == ITEM_TASK]
if len(item_df) > 2500:
    item_df = item_df.sample(2500, random_state=SEED)

train_user_df = pd.concat([user_df, item_df], ignore_index=True)
train_user_df = train_user_df.sample(frac=1, random_state=SEED).reset_index(drop=True)

print("Training rows:")
print("  user:", len(user_df))
print("  item:", len(item_df))
print("  total:", len(train_user_df))

# -------- Validation sets --------
VAL_USER_N = 100
VAL_ITEM_N = 100

val_user_df = user_df.sample(min(VAL_USER_N, len(user_df)), random_state=SEED).reset_index(drop=True)
val_item_df = item_df.sample(min(VAL_ITEM_N, len(item_df)), random_state=SEED).reset_index(drop=True)

@torch.no_grad()
def run_validation(model, val_user_df, val_item_df, device):
    model.eval()
    u_loss, i_loss = 0.0, 0.0
    u_cnt, i_cnt = 0, 0

    for i in range(len(val_user_df)):
        b = build_row_tensors(val_user_df.iloc[i], device)
        if not b["ok"]: continue
        out = model(b)
        u_loss += float(out["loss_user"].item())
        u_cnt += 1

    for i in range(len(val_item_df)):
        b = build_row_tensors(val_item_df.iloc[i], device)
        if not b["ok"]: continue
        out = model(b)
        # i_loss += float(out["loss_item"].item())
        i_cnt += 1

    model.train()
    return u_loss/max(1,u_cnt), i_loss/max(1,i_cnt)

# -------- Training --------
for epoch in range(start_epoch, EPOCHS):
    model.train()
    tot_loss = 0.0
    tot_user = 0.0
    tot_ok = 0

    row_start = start_row if epoch == start_epoch else 0
    pbar = tqdm(range(row_start, len(train_user_df)), desc=f"Epoch {epoch+1}/{EPOCHS}")

    for i in pbar:
        batch = build_row_tensors(train_user_df.iloc[i], device)
        if not batch["ok"]:
            continue

        out = model(batch)
        loss = out["loss"]
        if not torch.isfinite(loss):
            continue

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        tot_loss += float(loss.item())
        tot_user += float(out["loss_user"].item())
        tot_ok += 1

        if (i % 4500 == 0) or (i == len(train_user_df) - 1):
            torch.save(
                {
                    "epoch": epoch,
                    "row": i,
                    "model": model.state_dict(),
                    "optim": optimizer.state_dict()
                },
                ckpt_path(epoch, i)
            )

        if tot_ok > 0:
            pbar.set_postfix(avg_loss=tot_loss/tot_ok, avg_user=tot_user/tot_ok)

    # ---- Validation ----
    v_user, v_item = run_validation(model, val_user_df, val_item_df, device)

    print(
        f"Epoch {epoch+1}: "
        f"TrainLoss={tot_loss/max(1,tot_ok):.6f}, "
        f"TrainUser={tot_user/max(1,tot_ok):.6f} | "
        f"VAL User={v_user:.6f}, VAL Item={v_item:.6f}"
    )

    start_row = 0


FORCE resume from epoch034_row004500.pt (RNG ignored)
Training rows:
  user: 4507
  item: 2500
  total: 7007


Epoch 35/125:   0%|          | 0/2506 [00:00<?, ?it/s]

Epoch 35: TrainLoss=1.408829, TrainUser=0.128791 | VAL User=1.915685, VAL Item=0.000000


Epoch 36/125:   0%|          | 0/7007 [00:00<?, ?it/s]

Epoch 36: TrainLoss=1.373888, TrainUser=0.109652 | VAL User=2.116708, VAL Item=0.000000


...

Epoch 97: TrainLoss=0.971841, TrainUser=0.048505 | VAL User=0.369804, VAL Item=0.000000


Epoch 98/125:   0%|          | 0/7007 [00:00<?, ?it/s]

Epoch 98: TrainLoss=0.957790, TrainUser=0.039901 | VAL User=0.204267, VAL Item=0.000000


Epoch 99/125:   0%|          | 0/7007 [00:00<?, ?it/s]

KeyboardInterrupt: 

In [81]:
# =========================================================
# CELL 19 — Ranking Metrics
# =========================================================

def mrr_at_k(ranked, gold):
    for i, u in enumerate(ranked, 1):
        if u == gold:
            return 1.0 / i
    return 0.0

def ndcg_at_k(ranked, gold):
    for i, u in enumerate(ranked, 1):
        if u == gold:
            return 1.0 / math.log2(i + 1)
    return 0.0


In [83]:
# =========================================================
# CELL 20 — Evaluation (Next-speaker prediction, user-task only, 0–100 scale)
# =========================================================

K10 = 10
K5  = 5

model.eval()

mrrs = []
recall1 = []
hit5 = []
hit10 = []
ndcg5 = []
ndcg10 = []

n_eval = 0

def mrr_from_rank(rank):
    if rank is None:
        return 0.0
    return 1.0 / rank

def ndcg_from_rank(rank):
    if rank is None:
        return 0.0
    return 1.0 / math.log2(rank + 1)

# ---- restrict to next_user_prediction rows only ----
test_user_df = train_df[train_df["task"].astype(str).str.strip() == USER_TASK].reset_index(drop=True)
print("Evaluation rows (user only):", len(test_user_df))

with torch.no_grad():
    pbar = tqdm(range(len(test_user_df)), desc="Evaluating (user-only)")
    for i in pbar:
        batch = build_row_tensors(test_user_df.iloc[i], device)
        if not batch["ok"]:
            continue

        enc = model.encoder(
            e_seq=batch["e_seq"],
            action_onehot=batch["action_onehot"],
            action_labels=batch["action_labels"],
            node_labels=batch["node_labels"],
            task_labels=batch["task_labels"],
            q_text_emb=batch["q_text_emb"]
        )

        h = enc["h_final"]                              # (1, d)
        scores = model.user_ranker.scores_all(h)[0]     # (n_users,)
        top10 = torch.topk(scores, K10).indices.tolist()

        gold = int(batch["user_labels"].item())

        # -------- Rank of gold user --------
        rank = None
        for r, u in enumerate(top10, 1):
            if u == gold:
                rank = r
                break

        # -------- Metrics (0..1) --------
        mrrs.append(mrr_from_rank(rank))
        recall1.append(1.0 if rank == 1 else 0.0)
        hit5.append(1.0 if rank is not None and rank <= 5 else 0.0)
        hit10.append(1.0 if rank is not None and rank <= 10 else 0.0)
        ndcg5.append(ndcg_from_rank(rank) if rank is not None and rank <= 5 else 0.0)
        ndcg10.append(ndcg_from_rank(rank) if rank is not None and rank <= 10 else 0.0)

        n_eval += 1

        if n_eval > 0:
            pbar.set_postfix(
                MRR=100.0 * np.mean(mrrs),
                R1=100.0 * np.mean(recall1),
                H5=100.0 * np.mean(hit5),
                H10=100.0 * np.mean(hit10),
                N5=100.0 * np.mean(ndcg5),
                N10=100.0 * np.mean(ndcg10)
            )

print("======================================")
print("EVAL COUNT:", n_eval)
print(f"MRR@10    : {100.0 * float(np.mean(mrrs)):.4f}")
print(f"Recall@1  : {100.0 * float(np.mean(recall1)):.4f}")
print(f"Hit@5     : {100.0 * float(np.mean(hit5)):.4f}")
print(f"Hit@10    : {100.0 * float(np.mean(hit10)):.4f}")
print(f"NDCG@5    : {100.0 * float(np.mean(ndcg5)):.4f}")
print(f"NDCG@10   : {100.0 * float(np.mean(ndcg10)):.4f}")
print("======================================")


Evaluation rows (user only): 4507


Evaluating (user-only):   0%|          | 0/4507 [00:00<?, ?it/s]

EVAL COUNT: 4507
MRR@10    : 66.4056
Recall@1  : 53.9383
Hit@5     : 83.4702
Hit@10    : 90.7699
NDCG@5    : 69.9371
NDCG@10   : 72.3226


In [74]:
# =========================================================
# CELL 21 — Item / Node Retrieval (ITEM_TASK only, 0–100 scale)
# =========================================================

K = 15
SAMPLE_N = 200

# ---- Filter ITEM rows ----
item_eval_df = test_df[test_df["task"].astype(str).str.strip() == ITEM_TASK]

if len(item_eval_df) > SAMPLE_N:
    item_eval_df = item_eval_df.sample(SAMPLE_N, random_state=SEED)

item_eval_df = item_eval_df.reset_index(drop=True)

print("Item eval rows:", len(item_eval_df))

mrrs = []
ndcg5 = []
ndcg10 = []

def mrr(rank):
    return 0.0 if rank is None else 1.0 / rank

def ndcg(rank):
    return 0.0 if rank is None else 1.0 / math.log2(rank + 1)

model.eval()

with torch.no_grad():
    pbar = tqdm(range(len(item_eval_df)), desc="Evaluating ITEM retrieval")
    for i in pbar:
        batch = build_row_tensors(item_eval_df.iloc[i], device)
        if not batch["ok"]:
            continue

        enc = model.encoder(
            e_seq=batch["e_seq"],
            action_onehot=batch["action_onehot"],
            action_labels=batch["action_labels"],
            node_labels=batch["node_labels"],
            task_labels=batch["task_labels"],
            q_text_emb=batch["q_text_emb"]
        )

        h = enc["h_final"]   # (1, d)

        # ---- Score against ALL nodes ----
        scores = model.encoder.node_head(
            torch.cat([h, h, h], dim=-1)   # same fusion shape expected
        )[0]   # (|V|,)

        # ---- Top-15 predicted nodes ----
        topk = torch.topk(scores, K).indices.tolist()

        gold = int(batch["node_labels"][0, -1].item())

        # ---- Find rank ----
        rank = None
        for r, idx in enumerate(topk, 1):
            if idx == gold:
                rank = r
                break

        mrrs.append(mrr(rank))
        ndcg5.append(ndcg(rank) if rank is not None and rank <= 5 else 0.0)
        ndcg10.append(ndcg(rank) if rank is not None and rank <= 10 else 0.0)

        pbar.set_postfix(
            MRR=100.0 * np.mean(mrrs),
            N5=100.0 * np.mean(ndcg5),
            N10=100.0 * np.mean(ndcg10)
        )

print("======================================")
print("ITEM RETRIEVAL")
print("Samples :", len(item_eval_df))
print(f"MRR@15  : {100.0 * float(np.mean(mrrs)):.4f}")
print(f"NDCG@5  : {100.0 * float(np.mean(ndcg5)):.4f}")
print(f"NDCG@10 : {100.0 * float(np.mean(ndcg10)):.4f}")
print("======================================")


Item eval rows: 200


Evaluating ITEM retrieval:   0%|          | 0/200 [00:00<?, ?it/s]

ITEM RETRIEVAL
Samples : 200
MRR@15  : 0.0000
NDCG@5  : 0.0000
NDCG@10 : 0.0000


In [78]:
# =========================================================
# Show 1 example: gold vs predicted nodes
# =========================================================

if len(item_eval_df) > 0:
    ex = item_eval_df.iloc[0]
    ex_batch = build_row_tensors(ex, device)

    enc = model.encoder(
        e_seq=ex_batch["e_seq"],
        action_onehot=ex_batch["action_onehot"],
        action_labels=ex_batch["action_labels"],
        node_labels=ex_batch["node_labels"],
        task_labels=ex_batch["task_labels"],
        q_text_emb=ex_batch["q_text_emb"]
    )

    h = enc["h_final"]

    scores = model.encoder.node_head(torch.cat([h, h, h], dim=-1))[0]
    topk = torch.topk(scores, K)

    gold_node = ex_batch["tid"]
    gold_idx = node2idx[gold_node]


    print("\n========== ITEM EXAMPLE ==========")
    print("Gold last node:", gold_node, "(index:", gold_idx, ")")
    print("\nTop-15 predictions:")
    for rank, (idx, sc) in enumerate(zip(topk.indices.tolist(), topk.values.tolist()), 1):
        print(f"{rank:02d}. {idx2node[idx]}   score={sc:.4f}")
    print("=================================\n")



========== ITEM EXAMPLE ==========
Gold last node: d-42323 (index: 13061 )

Top-15 predictions:
01. d-62247   score=328.1195
02. d-30976   score=325.4914
03. d-73000   score=303.4415
04. d-46845   score=297.4776
05. d-12667   score=296.3928
06. d-42218   score=294.0350
07. d-98809   score=290.9251
08. d-110753   score=286.5270
09. d-38888   score=284.1537
10. d-37252   score=273.6181
11. d-33444   score=272.2932
12. d-64097   score=270.5183
13. d-53781   score=269.5417
14. d-112559   score=269.4366
15. d-11159   score=268.2677

